# Sanity check: quijotelike L1000-N128 vs abacuslike L2000-N256 halo catalogs (i=0..4)

Cross-volume consistency check for two box sizes of the same `fastpm_charm6` emulator.
Both use the same cosmology per lhid; volumes differ by 8x (1000³ vs 2000³ Mpc³/h³).
We expect the **number density** and **mass function shape** to agree — not raw counts.

Scale factor of interest: a = 0.666667 (quijotelike) / 0.666660 (abacuslike) ≈ z = 0.5.

Checks: (1) file structure & halo counts, (2) cosmology consistency per lhid,
(3) number density N/V, (4) halo mass function dN/dlogM/V,
(5) projected spatial distribution, (6) velocity PDFs, (7) concentration PDFs.

Run with the **`cmass`** conda env.

In [ ]:
import os
from os.path import join
import numpy as np
import h5py
import yaml
import matplotlib.pyplot as plt

BASEDIR_QUIJ = '/work/hdd/bdne/maho3/cmass-ili/quijotelike/fastpm_charm6/L1000-N128'
BASEDIR_ABAC = '/work/hdd/bdne/maho3/cmass-ili/abacuslike/fastpm_charm6/L2000-N256'
FIGDIR = '/u/maho3/git/ltu-gobig-notes/experiments/2026-06-12_sanity_quijotelike-fastpm_charm6_abacuslike-fastpm_charm6/figures'
L_QUIJ = 1000.0   # Mpc/h
L_ABAC = 2000.0   # Mpc/h
V_QUIJ = L_QUIJ ** 3
V_ABAC = L_ABAC ** 3
A_TARGET = 0.666667
LHIDS = [0, 1, 2, 3, 4, 10]

COLORS = {i: c for i, c in zip(LHIDS, ['C0', 'C1', 'C2', 'C3', 'C4', 'C5'])}
print('LHIDs:', LHIDS)
print(f'Quijotelike: L={L_QUIJ} Mpc/h,  V={V_QUIJ:.2e} (Mpc/h)^3')
print(f'Abacuslike:  L={L_ABAC} Mpc/h,  V={V_ABAC:.2e} (Mpc/h)^3  (8x larger)')

In [ ]:
# --- helpers ---------------------------------------------------------------
def find_snap_key(hdf5_path, a_target):
    """Return the snapshot group key closest to a_target."""
    with h5py.File(hdf5_path, 'r') as f:
        keys = list(f.keys())
    return min(keys, key=lambda k: abs(float(k) - a_target))


def load_halos(basedir, lhid):
    """Return (mass, pos, vel, conc, snap_key) for the closest snapshot to A_TARGET."""
    p = join(basedir, str(lhid), 'halos.h5')
    key = find_snap_key(p, A_TARGET)
    with h5py.File(p, 'r') as f:
        g = f[key]
        mass = g['mass'][:]
        pos = g['pos'][:]
        vel = g['vel'][:]
        conc = g['concentration'][:]
    return mass, pos, vel, conc, key


def load_cosmo(basedir, lhid):
    """Return [Omega_m, Omega_b, h, n_s, sigma8] from config.yaml."""
    with open(join(basedir, str(lhid), 'config.yaml')) as f:
        cfg = yaml.safe_load(f)
    return np.array(cfg['nbody']['cosmo'])


def mass_function(mass, volume, n_bins=30, m_min=None, m_max=None):
    """Compute dN/dlogM/V in logarithmic mass bins."""
    m_min = m_min or mass.min() * 0.9
    m_max = m_max or mass.max() * 1.1
    log_edges = np.linspace(np.log10(m_min), np.log10(m_max), n_bins + 1)
    counts, _ = np.histogram(np.log10(mass), bins=log_edges)
    log_centers = 0.5 * (log_edges[:-1] + log_edges[1:])
    dlogM = log_edges[1] - log_edges[0]
    dn_dlogM_dV = counts / (dlogM * volume)
    return 10 ** log_centers, dn_dlogM_dV

## 1. File structure & halo counts

In [ ]:
print(
    f"{'lhid':>6}  | {'suite':>14} | {'snap key':>10} | {'N_halos':>10} | n [h/Mpc]^3")
print('-' * 70)
for lhid in LHIDS:
    for label, basedir, V in [('quijotelike', BASEDIR_QUIJ, V_QUIJ),
                              ('abacuslike',  BASEDIR_ABAC, V_ABAC)]:
        p = join(basedir, str(lhid), 'halos.h5')
        assert os.path.isfile(p), f'MISSING: {p}'
        m, pos, vel, conc, key = load_halos(basedir, lhid)
        assert np.isfinite(m).all() and np.isfinite(
            pos).all(), f'non-finite values in {p}'
        assert (m > 0).all(), 'non-positive masses'
        print(f'{lhid:>6}  | {label:>14} | {key:>10} | {len(m):>10,} | {len(m)/V:.4e}')
print('\nAll files present, shapes finite, masses positive.')

## 2. Cosmology consistency per lhid

Both suites should use the **same** cosmology for the same lhid.
Any difference flags a misconfigured run.

In [ ]:
print(f"{'lhid':>6}  | match? | [Om, Ob, h, ns, s8]  (quijotelike)")
print('-' * 70)
for lhid in LHIDS:
    cq = load_cosmo(BASEDIR_QUIJ, lhid)
    ca = load_cosmo(BASEDIR_ABAC, lhid)
    match = np.allclose(cq, ca, rtol=1e-4)
    flag = 'OK ' if match else 'MISMATCH'
    print(f'{lhid:>6}  | {flag:^8} | {np.round(cq, 4).tolist()}')
    if not match:
        print(f'        |          | abacuslike: {np.round(ca, 4).tolist()}')
print('\nExpect: all lhids show OK.')

## 3. Number density N/V

Since both suites use the same cosmology per lhid and target the same scale factor,
the number density (N_halos / V_box) above a common mass cut should agree.
Quijotelike and abacuslike ratios should be close to 1.

In [ ]:
# Use the median minimum halo mass across quijotelike runs as a conservative cut.
M_MIN_CUT = np.median([load_halos(BASEDIR_QUIJ, i)[0].min() for i in LHIDS])
out_path = join(FIGDIR, 'number_density_report.txt')

lines = [
    f'Number-density mass cut: M > {M_MIN_CUT:.2e} Msun/h',
    '',
    f"{'lhid':>6} | {'n_quij [h/Mpc]^3':>20} | {'n_abac [h/Mpc]^3':>20} | ratio",
    '-' * 65,
]

for i in LHIDS:
    m_q, *_ = load_halos(BASEDIR_QUIJ, i)
    m_a, *_ = load_halos(BASEDIR_ABAC, i)
    n_q = (m_q > M_MIN_CUT).sum() / V_QUIJ
    n_a = (m_a > M_MIN_CUT).sum() / V_ABAC
    lines.append(f'{i:>6} | {n_q:>20.4e} | {n_a:>20.4e} | {n_q/n_a:.3f}')

lines.append('')
lines.append('Expect ratios close to 1.0.')

with open(out_path, 'w') as f:
    f.write('\n'.join(lines) + '\n')

print(f'Saved: {out_path}')
print('\n'.join(lines))

## 4. Halo mass function dN/dlogM/V

Each panel shows one lhid; quijotelike (solid) vs abacuslike (dashed) should trace
the same curve. Differences at the high-mass end are expected (sample variance —
the larger abacuslike box contains more rare massive halos).

In [ ]:
# Determine a global mass range for consistent binning across all runs.
all_masses_q = [load_halos(BASEDIR_QUIJ, i)[0] for i in LHIDS]
all_masses_a = [load_halos(BASEDIR_ABAC, i)[0] for i in LHIDS]
M_GLOBAL_MIN = min(m.min() for m in all_masses_q + all_masses_a)
M_GLOBAL_MAX = max(m.max() for m in all_masses_q + all_masses_a)

fig, axes = plt.subplots(2, len(LHIDS)//2, figsize=(
    4 * len(LHIDS)/2, 4.5*2), sharey=True)
axes = axes.flatten()
for ax, lhid, mq, ma in zip(axes, LHIDS, all_masses_q, all_masses_a):
    m_c, hmf_q = mass_function(
        mq, V_QUIJ, m_min=M_GLOBAL_MIN, m_max=M_GLOBAL_MAX)
    _,   hmf_a = mass_function(
        ma, V_ABAC, m_min=M_GLOBAL_MIN, m_max=M_GLOBAL_MAX)
    ax.loglog(m_c, hmf_q, color=COLORS[lhid], ls='-',  label='quijotelike')
    ax.loglog(m_c, hmf_a, color=COLORS[lhid], ls='--', label='abacuslike')
    ax.set_title(f'lhid {lhid}')
    ax.set_xlabel('M [Msun/h]')
    ax.legend(fontsize=8)
axes[0].set_ylabel('dN/dlogM/V [(Mpc/h)$^{-3}$]')
plt.suptitle(
    'Halo mass function (solid=quijotelike L1000, dashed=abacuslike L2000)', y=1.02)
plt.tight_layout()
plt.show()

fig.savefig(join(FIGDIR, 'halo_mass_function.jpg'))

## 5. Projected spatial distribution

2D projected positions of halos (thin slab in z). Should show cosmic web structure.
No obvious grid artefacts or empty patches. Shown for lhid=0.

In [ ]:
lhid = 0
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, (label, basedir, L) in zip(axes, [
        ('quijotelike  L=1000', BASEDIR_QUIJ, L_QUIJ),
        ('abacuslike   L=2000', BASEDIR_ABAC, L_ABAC)]):
    _, pos, *_ = load_halos(basedir, lhid)
    slab = (pos[:, 2] > 0.475 * L) & (pos[:, 2] < 0.525 * L)
    ax.scatter(pos[slab, 0], pos[slab, 1], s=0.5, alpha=0.5, rasterized=True)
    ax.set_xlim(0, L)
    ax.set_ylim(0, L)
    ax.set_title(f'{label}  (lhid {lhid},  N_slab={slab.sum():,d})')
    ax.set_xlabel('x [Mpc/h]')
    ax.set_ylabel('y [Mpc/h]')
    ax.set_aspect('equal')
plt.tight_layout()
plt.show()

fig.savefig(
    join(FIGDIR, f'halo_distribution_lhid{lhid}.jpg'), bbox_inches='tight')

In [ ]:
_, pos, *_ = load_halos(BASEDIR_ABAC, lhid)
slab = (pos[:, 2] > 0.475 * L) & (pos[:, 2] < 0.525 * L)

f, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.scatter(pos[slab, 0], pos[slab, 1], marker='.', s=0.5, alpha=1)
ax.plot(1000, 1000, 'kx')
ax.set_xlim(800, 1200)
ax.set_ylim(800, 1200)
ax.set(xlabel='x [Mpc/h]', ylabel='y [Mpc/h]')
f.savefig(join(
    FIGDIR, f'zoom_halo_distribution_lhid{lhid}.jpg'), bbox_inches='tight')

In [ ]:
# Stack thin-slab 2D histograms over all LHIDs, then average per suite.
NBINS = 384
SLAB_LO, SLAB_HI = 0.475, 0.525

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (label, basedir, Lbox) in zip(
    axes,
    [
        ('quijotelike  L=1000', BASEDIR_QUIJ, L_QUIJ),
        ('abacuslike   L=2000', BASEDIR_ABAC, L_ABAC),
    ],
):
    h_stack = []
    n_slab = []

    for lhid in range(50):
        _, pos, *_ = load_halos(basedir, lhid)
        slab = (pos[:, 2] > SLAB_LO * Lbox) & (pos[:, 2] < SLAB_HI * Lbox)

        H, xedges, yedges = np.histogram2d(
            pos[slab, 0], pos[slab, 1],
            bins=NBINS,
            range=[[0, Lbox], [0, Lbox]]
        )
        h_stack.append(H)
        n_slab.append(slab.sum())

    H_mean = np.mean(h_stack, axis=0)

    im = ax.imshow(
        np.log10(H_mean.T + 1.0),
        origin='lower',
        extent=[0, Lbox, 0, Lbox],
        aspect='equal',
        cmap='magma'
    )
    ax.set_title(
        f'{label}\nstacked over LHIDs=range(50)  '
        f'(mean N_slab={np.mean(n_slab):,.0f})'
    )
    ax.set_xlabel('x [Mpc/h]')
    ax.set_ylabel('y [Mpc/h]')
    fig.colorbar(im, ax=ax, label='log10(mean counts per pixel + 1)')

plt.tight_layout()
plt.show()

fig.savefig(
    join(FIGDIR, 'halo_distribution_stacked_mean_over_lhids.png'),
    dpi=400,
    bbox_inches='tight'
)

In [ ]:
FIGDIR

## 6. Velocity PDFs

Per-component velocity distributions should be zero-mean and roughly Gaussian.
Same cosmology per lhid → similar dispersions between quijotelike and abacuslike.
Solid lines = quijotelike, dashed = abacuslike.

In [ ]:
from matplotlib.lines import Line2D
fig, axes = plt.subplots(2, len(LHIDS)//2, figsize=(
    4 * len(LHIDS)/2, 4.5*2), sharey=True)
axes = axes.flatten()
for ax, lhid in zip(axes, LHIDS):
    _, _, vq, *_ = load_halos(BASEDIR_QUIJ, lhid)
    _, _, va, *_ = load_halos(BASEDIR_ABAC, lhid)
    bins = np.linspace(-2000, 2000, 120)
    for i, c in enumerate('xyz'):
        ax.hist(vq[:, i], bins=bins, histtype='step', density=True,
                color=f'C{i}', ls='-',  alpha=0.9)
        ax.hist(va[:, i], bins=bins, histtype='step', density=True,
                color=f'C{i}', ls='--', alpha=0.7)
    ax.set_title(f'lhid {lhid}')
    ax.set_xlabel('v [km/s]')
axes[0].set_ylabel('PDF')
# Manual legend
axes[0].legend(
    [Line2D([0], [0], color=f'C{i}') for i in range(3)] +
    [Line2D([0], [0], color='k', ls='-'), Line2D([0], [0], color='k', ls='--')],
    ['vx', 'vy', 'vz', 'quijotelike', 'abacuslike'], fontsize=7)
plt.suptitle('Halo velocity PDFs', y=1.02)
plt.tight_layout()
plt.show()

fig.savefig(
    join(FIGDIR, 'halo_velocity_pdfs.jpg'),
    bbox_inches='tight'
)

## 7. Concentration PDFs

NFW concentration should be positive and peaked around 5–15.
Same cosmology → similar distributions between the two suites.

In [ ]:
fig, axes = plt.subplots(2, len(LHIDS)//2, figsize=(
    4 * len(LHIDS)/2, 4.5*2), sharey=True)
axes = axes.flatten()
for ax, lhid in zip(axes, LHIDS):
    _, _, _, cq, _ = load_halos(BASEDIR_QUIJ, lhid)
    _, _, _, ca, _ = load_halos(BASEDIR_ABAC, lhid)
    bins = np.linspace(0, 50, 80)
    ax.hist(cq, bins=bins, histtype='step', density=True, color=COLORS[lhid],
            ls='-',  label='quijotelike')
    ax.hist(ca, bins=bins, histtype='step', density=True, color=COLORS[lhid],
            ls='--', label='abacuslike')
    ax.set_title(f'lhid {lhid}')
    ax.set_xlabel('concentration c')
    ax.legend(fontsize=8)
axes[0].set_ylabel('PDF')
plt.suptitle(
    'Halo concentration PDFs (solid=quijotelike, dashed=abacuslike)', y=1.02)
plt.tight_layout()
plt.show()

## 8. Halo power spectrum vs CAMB linear theory

Halos are NGP-painted onto a mesh; shot noise (L³/N_halos) is subtracted.
Quijotelike (solid) and abacuslike (dashed) should trace the same curve — same
cosmology, same emulator, same target scale factor. The grey dotted line is the
CAMB **linear matter** P(k); the halo P(k) lies above it at large scales by ~b²
and rises further at small scales from nonlinear clustering.

In [ ]:
import Pk_library as PKL
from cmass.nbody.tools import get_camb_pk

N_MESH_QUIJ = 128
N_MESH_ABAC = 256


def halo_pk(pos, boxsize, n_mesh, threads=4):
    """NGP-paint halo positions onto a mesh and compute P(k)."""
    delta, _ = np.histogramdd(pos, bins=n_mesh, range=[(0, boxsize)] * 3)
    delta = delta.astype(np.float32)
    delta /= delta.mean()
    delta -= 1.0
    pk_obj = PKL.Pk(delta, float(boxsize), axis=0, MAS='NGP',
                    verbose=False, threads=threads)
    return pk_obj.k3D, pk_obj.Pk[:, 0]


z = 1.0 / A_TARGET - 1.0

fig, axes = plt.subplots(
    2, len(LHIDS)//2, figsize=(4 * len(LHIDS)/2, 4.5*2), sharey=True)
axes = axes.flatten()
for ax, lhid in zip(axes, LHIDS):
    _, posq, *_ = load_halos(BASEDIR_QUIJ, lhid)
    _, posa, *_ = load_halos(BASEDIR_ABAC, lhid)
    cosmo = load_cosmo(BASEDIR_QUIJ, lhid)

    kq, pkq = halo_pk(posq, L_QUIJ, N_MESH_QUIJ)
    ka, pka = halo_pk(posa, L_ABAC, N_MESH_ABAC)
    shot_q = L_QUIJ**3 / len(posq)
    shot_a = L_ABAC**3 / len(posa)

    kcamb, pklin = get_camb_pk(kq.astype(float), *cosmo, z=z)

    ax.loglog(kq, pkq - shot_q,
              color=COLORS[lhid], ls='-',  label='quijotelike')
    ax.loglog(ka, pka - shot_a, color=COLORS[lhid], ls='--', label='abacuslike')
    ax.loglog(kcamb, pklin, color='grey', ls=':',
              alpha=0.8, label='CAMB linear')
    ax.axvline(np.pi * N_MESH_QUIJ / L_QUIJ, color='grey',
               ls='--', alpha=0.4, label='Nyquist')
    ax.set_title(f'lhid {lhid}')
    ax.set_xlabel('k [h/Mpc]')
    ax.legend(fontsize=7)
axes[0].set_ylabel('P(k) [(Mpc/h)$^3$]')
plt.suptitle(
    'Halo P(k) shot-noise subtracted  (solid=quijotelike, dashed=abacuslike, dotted=CAMB)',
    y=1.02)
plt.tight_layout()
plt.show()

fig.savefig(join(FIGDIR, 'halo_power_spectra.jpg'), bbox_inches='tight')